In [ ]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("../../data/DieCasting_Quality_Raw_Data.csv", skiprows=1)
df["defect"] = df[df.columns[31:]].max(axis=1).astype(bool).astype(int)
df = df[df.columns[:31].tolist() + ["defect"]]
df.groupby("defect").size()

In [ ]:
# Split data into features (X) and target (y)
X = df.drop("defect", axis=1)
y = df["defect"]


# First split into train+val (80%) and test (20%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Then split train+val into train (60%) and val (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42
)

# Create dataset for LightGBM
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val)

# Set parameters for LightGBM
params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "num_leaves": 31,
    "learning_rate": 0.05,
    "feature_fraction": 0.9,
    "is_unbalance": True,
}

# Train model
num_round = 100
lgb_model = lgb.train(params, train_data, num_round, valid_sets=[train_data, val_data])


In [ ]:
from sklearn.metrics import classification_report

# Evaluate on validation set
val_predictions = (lgb_model.predict(X_val) > 0.5).astype(int)
print("\nValidation Set Performance:")
print(classification_report(y_val, val_predictions))

# Evaluate on test set
test_predictions = (lgb_model.predict(X_test) > 0.5).astype(int)
print("\nTest Set Performance:")
print(classification_report(y_test, test_predictions))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, test_predictions)